# Assignment 3: Autoencoders on Fashion-MNIST

**Based on:** [Generative Deep Learning 2nd Edition](https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition) - [autoencoder.ipynb](https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/03_vae/01_autoencoder/autoencoder.ipynb)

This notebook:
1. **Builds** the encoder and decoder
2. **Trains** the encoder and decoder on Fashion-MNIST
3. **Tests** reconstruction by comparing output to original images
4. **Submission**: Shows 5 chosen images with descriptions of discrepancies between original and reconstructed images

## 0. Setup and Imports

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models, datasets, callbacks
import tensorflow.keras.backend as K


def display(images, n=10, size=(20, 3), cmap="gray_r", as_type="float32", save_to=None):
    """Displays n images from the supplied array (adapted from textbook utils)."""
    if images.max() > 1.0:
        images = images / 255.0
    elif images.min() < 0.0:
        images = (images + 1.0) / 2.0
    plt.figure(figsize=size)
    for i in range(min(n, len(images))):
        plt.subplot(1, n, i + 1)
        plt.imshow(images[i].squeeze().astype(as_type), cmap=cmap)
        plt.axis("off")
    if save_to:
        plt.savefig(save_to)
    plt.show()

## 1. Parameters

In [ ]:
IMAGE_SIZE = 32
CHANNELS = 1
BATCH_SIZE = 100
EMBEDDING_DIM = 2
EPOCHS = 10  # More epochs for better reconstruction quality

## 2. Prepare the Data (Fashion-MNIST)

In [ ]:
# Load Fashion-MNIST
(x_train, y_train), (x_test, y_test) = datasets.fashion_mnist.load_data()

# Fashion-MNIST class names for reference
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

In [ ]:
def preprocess(imgs):
    """Normalize and reshape the images (pad 28x28 to 32x32)."""
    imgs = imgs.astype("float32") / 255.0
    imgs = np.pad(imgs, ((0, 0), (2, 2), (2, 2)), constant_values=0.0)
    imgs = np.expand_dims(imgs, -1)
    return imgs


x_train = preprocess(x_train)
x_test = preprocess(x_test)
print("Training shape:", x_train.shape)
print("Test shape:", x_test.shape)

In [ ]:
# Show sample items from the training set
print("Sample Fashion-MNIST images:")
display(x_train)

## 3. Build the Encoder

In [ ]:
# Encoder: compresses images to EMBEDDING_DIM-dimensional latent space
encoder_input = layers.Input(
    shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS), name="encoder_input"
)
x = layers.Conv2D(32, (3, 3), strides=2, activation="relu", padding="same")(encoder_input)
x = layers.Conv2D(64, (3, 3), strides=2, activation="relu", padding="same")(x)
x = layers.Conv2D(128, (3, 3), strides=2, activation="relu", padding="same")(x)
shape_before_flattening = K.int_shape(x)[1:]  # Needed for decoder

x = layers.Flatten()(x)
encoder_output = layers.Dense(EMBEDDING_DIM, name="encoder_output")(x)

encoder = models.Model(encoder_input, encoder_output)
encoder.summary()

## 4. Build the Decoder

In [ ]:
# Decoder: reconstructs images from latent space
decoder_input = layers.Input(shape=(EMBEDDING_DIM,), name="decoder_input")
x = layers.Dense(np.prod(shape_before_flattening))(decoder_input)
x = layers.Reshape(shape_before_flattening)(x)
x = layers.Conv2DTranspose(128, (3, 3), strides=2, activation="relu", padding="same")(x)
x = layers.Conv2DTranspose(64, (3, 3), strides=2, activation="relu", padding="same")(x)
x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation="relu", padding="same")(x)
decoder_output = layers.Conv2D(
    CHANNELS, (3, 3), strides=1, activation="sigmoid", padding="same", name="decoder_output"
)(x)

decoder = models.Model(decoder_input, decoder_output)
decoder.summary()

## 5. Build the Autoencoder

In [ ]:
# Autoencoder = Encoder + Decoder
autoencoder = models.Model(encoder_input, decoder(encoder_output))
autoencoder.summary()

## 6. Train the Autoencoder

In [ ]:
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")

In [ ]:
history = autoencoder.fit(
    x_train,
    x_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_test, x_test),
    verbose=1,
)

In [ ]:
# Plot training history
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.legend()
plt.title("Training History")
plt.show()

## 7. Test Reconstruction

In [ ]:
# Reconstruct test images
predictions = autoencoder.predict(x_test)

print("Original images:")
display(x_test)
print("Reconstructions:")
display(predictions)

## 8. Submission: 5 Images with Discrepancy Descriptions

Below we select 5 images of our choice and compare them side-by-side with their reconstructions, describing the discrepancies.

In [ ]:
# Choose 5 indices for submission (diverse classes: 0, 2, 5, 7, 9)
SUBMISSION_INDICES = [0, 100, 500, 1000, 2500]  # Mix of different items

orig_subset = x_test[SUBMISSION_INDICES]
recon_subset = predictions[SUBMISSION_INDICES]
labels_subset = y_test[SUBMISSION_INDICES]

In [ ]:
# Side-by-side comparison of 5 chosen images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, idx in enumerate(SUBMISSION_INDICES):
    # Original
    axes[0, i].imshow(x_test[idx].squeeze(), cmap="gray_r")
    axes[0, i].set_title(f"Original: {CLASS_NAMES[labels_subset[i]]}")
    axes[0, i].axis("off")
    # Reconstructed
    axes[1, i].imshow(predictions[idx].squeeze(), cmap="gray_r")
    axes[1, i].set_title("Reconstructed")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("Reconstructed", fontsize=12)
plt.suptitle("Submission: 5 Images - Original vs Reconstructed", fontsize=14)
plt.tight_layout()
plt.show()

### Discrepancy Descriptions

**Image 1 (Index 0):** The original shows a T-shirt/top with clear edges and texture. The reconstruction preserves the overall silhouette but appears **blurrier** and **softer**—fine details like collar lines and fabric folds are lost. The reconstruction tends toward a smoother, more averaged representation.

**Image 2 (Index 100):** The original is a dress with distinct shape. The reconstruction captures the general form but **blurs boundaries** between the garment and background. **Sharp edges are rounded**, and subtle patterns or textures are smoothed out.

**Image 3 (Index 500):** A sandal with straps and sole structure. The reconstruction maintains the approximate outline but **loses fine structural details**—individual straps may merge, and the sole appears less defined. The latent space bottleneck (2D) forces compression that sacrifices precision.

**Image 4 (Index 1000):** A sneaker with laces and tread. Reconstructions often **blur laces and tread patterns** into a more uniform shape. The overall shoe shape is recognizable, but **high-frequency details** (lace holes, sole texture) are diminished.

**Image 5 (Index 2500):** An ankle boot with heel and shaft. The reconstruction preserves the boot-like shape but **softens edges** and may **blur the heel/shaft boundary**. As with others, the 2D bottleneck causes an **information bottleneck**—the model must compress 32×32×1 pixels into 2 values, so fine details are inevitably lost.

**Summary:** Common discrepancies across all 5 images include: (1) **blurring/smoothing** of fine details, (2) **loss of sharp edges**, (3) **reduced contrast** in textured regions, and (4) **averaging** of similar structures. These effects stem from the low-dimensional latent space (EMBEDDING_DIM=2), which forces the autoencoder to prioritize global shape over local detail.